# Sanity run: single-turn simulation (tiny batch)

Quick check to ensure seeds/strategy/follow-up are non-empty and emotion is preserved after recent robustness/seeding fixes.

In [ ]:
from pathlib import Path
import json
from dynamic_conversation import SingleTurnSimulator, SimulationConfig, ResponseStrategy


from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI')

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

cfg = SimulationConfig(
    model="gpt-4o-mini",
    max_tokens=800,
    temperature=1.0,
    brevity_hint="Reply in 1-2 sentences, keep emotion visible."
)

sim = SingleTurnSimulator(
    use_gpu=False,
    config=cfg,
    prompt_for_key=True,
)

emotions = ["anger", "joy"]
strategies = [ResponseStrategy.VALIDATE, ResponseStrategy.GUIDE]

df = sim.run_batch(
    emotions=emotions,
    strategies=strategies,
    runs_per_pair=1,
    style_modifier="concise and emotionally attuned",
    use_llm_seed=True,
    include_baseline=True,
    save_csv=results_dir / "sanity_single_turn.csv",
    save_heatmap=results_dir / "sanity_single_turn_heatmap.png",
)

df

In [ ]:
# Inspect meta/log to confirm no failures
meta_path = results_dir / "sanity_single_turn.meta.json"
log_path = results_dir / "sanity_single_turn.log"
if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    print("Successes:", meta.get("success_count"), "Failures:", meta.get("failure_count"))
    print("Config:", meta.get("config"))
else:
    print("Meta not found")

if log_path.exists():
    print("\nFailure log:")
    print(log_path.read_text())
else:
    print("No failure log")
